In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Generate synthetic data
np.random.seed(42)
n_samples, n_features = 300, 10

undamaged = np.random.normal(0, 1, (n_samples // 2, n_features))
damaged = np.random.normal(2, 1, (n_samples // 2, n_features))

X = np.vstack((undamaged, damaged))
y = np.array([0] * (n_samples // 2) + [1] * (n_samples // 2))

# Split and standardize
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# AIS Classifier
class AISClassifier:
    def __init__(self, clones=10, iters=20, selected=5, mutation=0.05):
        self.clones = clones
        self.iters = iters
        self.selected = selected
        self.mutation = mutation
        self.memory = []

    def _distance(self, a, b):
        return np.linalg.norm(a - b)

    def _mutate(self, a):
        return a + self.mutation * np.random.randn(*a.shape)

    def fit(self, X, y):
        self.memory = []
        for label in np.unique(y):
            samples = X[y == label]
            pop = samples[np.random.choice(len(samples), self.selected, replace=False)]

            for _ in range(self.iters):
                clones = np.vstack([self._mutate(a) for a in pop for _ in range(self.clones)])
                scores = [np.mean([self._distance(c, x) for x in samples]) for c in clones]
                pop = clones[np.argsort(scores)[:self.selected]]

            self.memory.append((label, pop))

    def predict(self, X):
        preds = []
        for x in X:
            dists = [(min(self._distance(a, x) for a in mem), label) for label, mem in self.memory]
            preds.append(min(dists)[1])
        return np.array(preds)

# Train and evaluate
clf = AISClassifier()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")


Accuracy: 100.00%
